In [0]:
spark

SparkSession - hive 
 
 
 SparkContext 

 Spark UI 

 
 Version 
 v3.3.2 
 Master 
 local[8] 
 AppName 
 Databricks Shell

In [0]:
# Performing necessary imports required for the project
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:

# Creating Spark Session  with Appname
spark = SparkSession.builder.appName("Health Care Data Engineering Spark Project").getOrCreate()

# getOrCreate() this will get the app if created or else it will create a new session

In [0]:
# Creating Structural Schema
# To set a default schema for every data that will be loaded

# Struct Schema for conditions
conditions_schema = StructType([
    StructField("start", TimestampType(),True),
    StructField("stop", TimestampType(),True),
    StructField("patient", StringType(),True),
    StructField("encounter", StringType(),True),
    StructField("code", StringType(),True),
    StructField("description", StringType(),True),
])

encounters_schema = StructType([
    StructField("id", StringType(), True),
    StructField("start", TimestampType(), True),
    StructField("stop", TimestampType(), True),
    StructField("patient", StringType(), True),
    StructField("organization", StringType(), True),
    StructField("provider", StringType(), True),
    StructField("payer", StringType(), True),
    StructField("encounterclass", StringType(), True),
    StructField("code", IntegerType(), True),
    StructField("description", StringType(), True),
    StructField("base_encounter_cost", DoubleType(), True),
    StructField("total_claim_cost", DoubleType(), True),
    StructField("payer_coverage", DoubleType(), True),
    StructField("reasoncode", StringType(), True)
])

# Schema for immunizations_df
immunizations_schema = StructType([
    StructField("date", TimestampType(), True),
    StructField("patient", StringType(), True),
    StructField("encounter", StringType(), True),
    StructField("code", IntegerType(), True),
    StructField("description", StringType(), True)
])

# Schema for patients_df
patients_schema = StructType([
    StructField("id", StringType(), True),
    StructField("birthdate", DateType(), True),
    StructField("deathdate", DateType(), True),
    StructField("ssn", StringType(), True),
    StructField("drivers", StringType(), True),
    StructField("passport", StringType(), True),
    StructField("prefix", StringType(), True),
    StructField("first", StringType(), True),
    StructField("last", StringType(), True),
    StructField("suffix", StringType(), True),
    StructField("maiden", StringType(), True),
    StructField("marital", StringType(), True),
    StructField("race", StringType(), True),
    StructField("ethnicity", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("birthplace", StringType(), True),
    StructField("address", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("county", StringType(), True),
    StructField("fips", IntegerType(), True),
    StructField("zip", IntegerType(), True),
    StructField("lat", DoubleType(), True),
    StructField("lon", DoubleType(), True),
    StructField("healthcare_expenses", DoubleType(), True),
    StructField("healthcare_coverage", DoubleType(), True),
    StructField("income", IntegerType(), True),
    StructField("mrn", IntegerType(), True)
])

In [0]:

def load_csv(spark, schema, file_path):
    """Loads  CSV file from S3 into a Spark DataFrame."""
    return spark.read.schema(schema).option("header", "true").csv(file_path)


# CSV File name paths from AWS S3 Bucket and Pre-defined schemas
csv_files = [
    ("conditions", "s3://health-care-data-bucket/conditions.csv", conditions_schema),
    ("encounters", "s3://health-care-data-bucket/encounters.csv", encounters_schema),
    ("immunizations", "s3://health-care-data-bucket/immunizations.csv", immunizations_schema),
    ("patients", "s3://health-care-data-bucket/patients.csv", patients_schema)
]

# Load CSV files
dataframes = {}

# Running loop for each file
for name, path, schema in csv_files:
    dataframes[name] = load_csv(spark, schema, path)

# Variables to access DataFrames
conditions_df = dataframes["conditions"]
encounters_df = dataframes["encounters"]
immunizations_df = dataframes["immunizations"]
patients_df = dataframes["patients"]






In [0]:
# Data Cleaning and Transformation of Patient Dataframe

# standardizing column marital
clean_patients_df = patients_df.withColumn('marital', when(patients_df.marital == 'M', 'Married').when(patients_df.marital == 'S', 'Single').when(patients_df.marital == 'D', 'Divorced').otherwise('n/a'))

new_clean_patients_df = clean_patients_df.withColumn('gender', when(patients_df.gender == 'M', 'Male').when(patients_df.gender == 'F', 'Female').otherwise('n/a'))

#Renaming Columns with user friendly names
new_patients_df = new_clean_patients_df.withColumnRenamed("id","patient_id")\
    .withColumnRenamed("first","first_name")\
    .withColumnRenamed("last","last_name")\
    .withColumnRenamed("lat","latitude")\
    .withColumnRenamed("lon","longitutde")

new_patients_df.display()


patient_id,birthdate,deathdate,ssn,drivers,passport,prefix,first_name,last_name,suffix,maiden,marital,race,ethnicity,gender,birthplace,address,city,state,county,fips,zip,latitude,longitutde,healthcare_expenses,healthcare_coverage,income,mrn
73d3ebe3-e656-b9de-fd61-88d370a86d51,1985-09-15,null,999-54-9859,S99986862,X21012070X,Mrs.,Hedy,Von,NULL,Schoen,Married,white,nonhispanic,Female,Worcester Massachusetts US,1012 Lueilwitz Trail Unit 2,Reading,Massachusetts,Middlesex County,25017,1867,42.52496000289047,-71.1210503282821,8522.61,261550.12,8402,1
cc53e99a-6715-603f-b074-eea93fe11e20,1961-07-26,2019-08-17,999-19-2105,S99922222,X66407833X,Mrs.,Adelina,Treutel,NULL,Miller,Divorced,white,nonhispanic,Female,Scituate Massachusetts US,622 Kilback Loaf,Springfield,Massachusetts,Hampden County,25013,1104,42.09394120412677,-72.5707445727102,824192.36,31979.82,42514,2
15e61a30-618d-4b20-dd5d-5dc627fa6e4c,2001-12-27,null,999-44-5853,S99929170,X68583323X,Ms.,Artie,Cronin,NULL,NULL,n/a,white,nonhispanic,Female,Brockton Massachusetts US,763 Tillman Junction,Worcester,Massachusetts,Worcester County,25027,1604,42.291968201261824,-71.81152962280308,32502.1,767484.6,638066,3
32a2188a-132e-4fd4-1a0e-3d532f4c4ea8,1958-12-17,null,999-78-8436,S99914103,X28243259X,Mrs.,Lauryn,Wisozk,NULL,Senger,Married,white,hispanic,Female,Fitchburg Massachusetts US,797 Lemke Bypass Unit 25,Boston,Massachusetts,Suffolk County,25025,2121,42.25729277762842,-71.08346502522105,437107.67,933931.36,147695,4
4adcad4e-1aa5-5601-4107-55953f484703,1962-06-26,null,999-72-3919,S99998121,X76753902X,Mrs.,Dorthea,Reichel,NULL,McDermott,Married,black,nonhispanic,Female,Fall River Massachusetts US,865 Simonis Highlands Suite 83,Braintree,Massachusetts,Norfolk County,25021,2184,42.23661190444311,-71.00959511002986,254423.25,590863.07,122035,5
52de8610-de40-203d-4bd6-cf06141fa864,1975-11-08,null,999-27-4817,S99922577,X50092027X,Mrs.,Sherita,Orn,NULL,Huels,Divorced,white,nonhispanic,Female,Northborough Massachusetts US,508 Haag Lock,Ludlow,Massachusetts,Hampden County,null,0,42.135095795514815,-72.44664055250006,556114.94,11443.34,55707,6
154290c8-6729-fe33-d3b6-a66f04bb939e,1964-10-02,null,999-12-6101,S99929848,X1401129X,Ms.,Elenora,Raynor,NULL,NULL,Single,white,nonhispanic,Female,Fairhaven Massachusetts US,608 Lind Forge Apt 24,East Sandwich,Massachusetts,Barnstable County,25001,2537,41.76856882290082,-70.45565220982637,22782.33,958551.12,261849,7
10d940ae-7114-8bcf-50f9-2bfd5354ff41,1962-05-17,null,999-95-9039,S99962381,X43169098X,Mrs.,Rebeca,Mayer,NULL,Beatty,Married,white,nonhispanic,Female,Randolph Massachusetts US,797 Monahan Divide Unit 44,Methuen,Massachusetts,Essex County,25009,1844,42.70238540126656,-71.19779376499118,654622.19,270730.9,60975,8
ac63da4a-893a-1d3c-aa93-fb78c8da3d95,1971-04-21,null,999-34-2266,S99975964,X17674028X,Mrs.,Casie,Hilpert,NULL,Friesen,Married,white,nonhispanic,Female,Billerica Massachusetts US,715 Mitchell Plaza,Dover,Massachusetts,Norfolk County,25021,2030,42.252071748674375,-71.26056668178677,312357.59,1001217.44,247569,9
bcb267a1-09ab-7082-96bd-81b867c68da7,1983-10-04,null,999-83-2417,S99950708,X82674718X,Mrs.,Arline,Jenkins,NULL,Beatty,Married,white,nonhispanic,Female,Westford Massachusetts US,570 Ryan Station,Hanover,Massachusetts,Plymouth County,null,0,42.15780614923903,-70.8831098081869,11404.46,657952.08,17204,10


In [0]:
# Data Cleaning and Transformation of immunization Dataframe

# Renaming the columns with friendly names
new_immunization_df = immunizations_df.withColumn("date", immunizations_df["date"].cast("date")).withColumnRenamed('patient','vaccined_patient_id')\
     .withColumnRenamed("encounter","encounter_id")\
     .withColumnRenamed("code","vaccine_code")

new_immunization_df.limit(10).display()

date,vaccined_patient_id,encounter_id,vaccine_code,description
2011-07-27,cc53e99a-6715-603f-b074-eea93fe11e20,ac02e16e-618c-e24a-b1d8-d4287503e731,5301,Herpes Zoster Vaccine (Live)
2011-07-27,cc53e99a-6715-603f-b074-eea93fe11e20,ac02e16e-618c-e24a-b1d8-d4287503e731,5302,Seasonal Flu Vaccine
2013-12-09,73d3ebe3-e656-b9de-fd61-88d370a86d51,2bb23033-f5e9-024e-35fb-9d939ca69f18,5302,Seasonal Flu Vaccine
2014-12-15,73d3ebe3-e656-b9de-fd61-88d370a86d51,570d34ce-6c5d-bbb0-669b-66abbd3fd112,5302,Seasonal Flu Vaccine
2012-08-01,cc53e99a-6715-603f-b074-eea93fe11e20,fa522491-1733-3943-5af4-944767376175,5301,Herpes Zoster Vaccine (Live)
2012-08-01,cc53e99a-6715-603f-b074-eea93fe11e20,fa522491-1733-3943-5af4-944767376175,5302,Seasonal Flu Vaccine
2012-08-01,cc53e99a-6715-603f-b074-eea93fe11e20,fa522491-1733-3943-5af4-944767376175,5303,"Five doses of tetanus toxoid, preservative-free and adsorbed, for adults."
2013-08-07,cc53e99a-6715-603f-b074-eea93fe11e20,4b9cc4c9-d58f-a065-722c-7e8b6ebd4a84,5302,Seasonal Flu Vaccine
2015-12-21,73d3ebe3-e656-b9de-fd61-88d370a86d51,a4fdcef4-194e-54dd-ac51-dc1ffa85eb2d,5302,Seasonal Flu Vaccine
2016-12-26,73d3ebe3-e656-b9de-fd61-88d370a86d51,34c25b3f-0d79-4506-243a-e1a0c89ad908,5302,Seasonal Flu Vaccine


In [0]:

#Quality Check encounters
# Getting the Date data and time data seperately

#Start Date and Time Seperation
clean_start_time_df = encounters_df.withColumn("start_time", date_format('start', 'HH:mm:ss') )
clean_start_date_df = clean_start_time_df.withColumn("start", encounters_df['start'].cast('date'))

#Stop Date and Time Seperation 
clean_stop_time_df = clean_start_date_df.withColumn("end_time", date_format('stop', 'HH:mm:ss'))
clean_end_date_df = clean_stop_time_df.withColumn("stop", encounters_df['stop'].cast('date'))

# Renaming the columns with friendly names
new_encounters_df = clean_end_date_df.withColumnRenamed('id','encounter_id')\
    .withColumnRenamed('start','start_date')\
    .withColumnRenamed('stop','stop_date')\
    .withColumnRenamed('patient','patient_id')\
    .withColumnRenamed('code','encounter_code')

new_encounters_df.limit(20).display()

encounter_id,start_date,stop_date,patient_id,organization,provider,payer,encounterclass,encounter_code,description,base_encounter_cost,total_claim_cost,payer_coverage,reasoncode,start_time,end_time
be86bb53-1982-c56d-ee22-ac961787aa0c,2018-04-07,2018-04-07,bb8d3c0d-78f6-747e-bd03-9de9efd98a21,9d0e702d-50a0-3f4c-9126-0951d560fd4b,179a5ef5-b06b-39c2-82f8-b552b709eb3c,8fa6c185-e44e-3e34-8bd8-39be8694f4ce,ambulatory,1032,Hospital Encounter with Problem,85.55,85.55,0.0,NULL,21:17:11,21:32:11
a185944c-70c1-3fdf-073a-9ba86d29606d,2019-10-25,2019-10-25,bb8d3c0d-78f6-747e-bd03-9de9efd98a21,9d0e702d-50a0-3f4c-9126-0951d560fd4b,179a5ef5-b06b-39c2-82f8-b552b709eb3c,8fa6c185-e44e-3e34-8bd8-39be8694f4ce,ambulatory,1032,Hospital Encounter with Problem,85.55,85.55,0.0,NULL,12:17:11,12:32:11
67deb48d-6bc0-8142-189d-1f10abc7c6bc,2018-12-13,2018-12-13,a3a96fd1-3638-41d3-72dc-efc248f2b887,217cb6f6-e822-3831-9d9d-ffa104971042,7077be2a-5b48-35f1-98b8-5e5b5a42343b,26aab0cd-6aba-3e1b-ac5b-05c8867e762c,ambulatory,1032,Hospital Encounter with Problem,85.55,85.55,0.0,NULL,23:11:54,23:26:54
de112f81-cfa9-ab77-8c56-e7ee074a0abe,2022-10-09,2022-10-09,b1a0a29e-113d-903c-6cef-016235be98e8,ae3eab22-8868-37bb-9a59-2b8bfe14bf34,1e9fb93b-b6e1-3e44-b1d6-cb0c323bee95,d31fccc3-1767-390d-966a-22a5156f4219,ambulatory,1032,Hospital Encounter with Problem,85.55,85.55,0.0,NULL,01:53:22,02:08:22
ea6202ee-7152-3e80-d779-5a53fe351f19,2020-05-09,2020-05-09,bd603d4c-3093-2104-e5f0-360cb08b7536,901c2d40-1ca3-3879-9a20-c663b8adc0a9,ec66f0b4-c703-33ad-ac54-5c1480a450de,b046940f-1664-3047-bca7-dfa76be352a4,ambulatory,1032,Hospital Encounter with Problem,85.55,85.55,21.71,NULL,13:34:31,13:49:31
1ace1153-4699-5c0c-4f3a-58cb3c2ec648,2022-03-05,2022-03-05,2afea9cf-f03f-7408-0535-d640b003c339,5018c664-e283-30eb-932a-529d9a19b3b5,0204406f-f2dd-35c4-8945-a4d788d4a287,b046940f-1664-3047-bca7-dfa76be352a4,ambulatory,1032,Hospital Encounter with Problem,85.55,85.55,0.0,NULL,08:43:58,08:58:58
74c235c9-59c0-94b7-ec80-108da7a1165f,2017-08-31,2017-08-31,0fab3069-e6e1-33a8-c21c-580c6cc989f4,b6eeaaf7-1683-3bcb-b6ee-81ce304636ef,9deecdc7-972f-378a-8659-6981b6cd3bd4,e03e23c9-4df1-3eb6-a62d-f70f02301496,ambulatory,1032,Hospital Encounter with Problem,85.55,85.55,0.0,NULL,17:23:50,17:38:50
8f45763c-ea2d-e462-7f94-5b0a95247b8e,2022-05-12,2022-05-12,4361f740-2bce-01eb-00d1-5b3344ae7464,0fedae9f-701f-3317-9b2f-69aea2202cdc,abdf12f9-2a02-3ca4-8b36-673a675a6771,b046940f-1664-3047-bca7-dfa76be352a4,ambulatory,1032,Hospital Encounter with Problem,85.55,85.55,0.0,NULL,08:13:56,08:28:56
7eedb204-e714-7944-8fe4-35bcc593daf1,2016-12-18,2016-12-18,dc5fe737-c79e-66d7-b834-cec1ae473dab,20df65a4-7567-3066-b680-0f71b0c31d38,13bd9bb2-e784-35c1-8a3d-cb10f8488571,0133f751-9229-3cfd-815f-b6d4979bdd6a,ambulatory,1032,Hospital Encounter with Problem,85.55,85.55,85.55,NULL,06:40:37,06:55:37
0f640694-f5f5-ef8e-a582-c2b2ceaf8e6a,2019-07-28,2019-07-28,a9562614-9c3a-6246-a12e-10cff583a743,a537b406-fdfa-36b4-84da-12512e7e6c63,e714484d-1a16-3a8c-98fe-842ff9655cd5,26aab0cd-6aba-3e1b-ac5b-05c8867e762c,ambulatory,1032,Hospital Encounter with Problem,85.55,85.55,0.0,NULL,22:24:47,22:39:47


In [0]:
# Cleaning and Standardizing Condition Data frame

new_conditions_df = conditions_df.withColumn('start', conditions_df['start'].cast('date')).withColumn('stop', conditions_df['stop'].cast('date')).withColumnRenamed('start','start_date')\
        .withColumnRenamed('stop','stop_date')\
        .withColumnRenamed('patient','patient_id')\
        .withColumnRenamed('encounter','encounter_id')

new_conditions_df.limit(10).display()

start_date,stop_date,patient_id,encounter_id,code,description
1953-06-03,null,531997f3-3373-058b-6315-bab2b9f05502,b67eb71d-01d4-e225-7203-8d226d9079de,473.8,Other chronic sinusitis
1952-03-12,null,12556183-9867-11ec-e7d6-4cc149e28bf6,56c1ae8b-c774-0259-9291-782e502027e4,473.9,Unspecified sinusitis (chronic)
1957-10-09,null,82e05bd5-78f8-2fe4-caba-33c4afdcc9ef,5f37acfc-84c0-7bac-284f-2b8fd9ef61ac,V60.9,Unspecified housing or economic circumstance
1954-02-13,1956-02-25,63dc3aeb-c77c-441a-ffbd-a4d6985fbd77,4c822b22-96ee-f968-e29a-0227af97c084,V62.89,"Other psychological or physical stress, not elsewhere classified"
1951-09-13,null,d70308e9-5bce-1154-cefe-27f61aede16d,9967cce2-0a85-9dcd-92f1-52aa0fee7d7a,541,"Appendicitis, unqualified"
1959-09-29,null,59ce02b3-e5d8-da4c-2f70-b7018a853333,a268dba0-dd76-3c9f-8afc-b6861e9e7c37,473.8,Other chronic sinusitis
1958-10-01,1959-04-30,5799e37d-3ae4-bd7b-61e2-8f4d6fb90956,f541a806-dd26-a3cd-478d-d57274d7a996,V22.2,"Pregnant state, incidental"
1959-12-22,null,9b0886ff-99fd-c2db-7d0c-7c25f8ee8636,b2690c76-b9e7-e737-57fe-33106b48ffb0,542,Other appendicitis
1949-02-15,null,dc415ea3-e2d6-3808-ec78-79460d1ad67e,97b70d55-5289-67c8-73b8-f612963b7f75,V85.30,"Body Mass Index 30.0-30.9, adult"
1955-09-30,1956-05-04,5fbb2718-d088-9217-abe8-cefb053869ac,ea804eed-1031-20cd-9ea4-f86cdd2e9b53,V62.89,"Other psychological or physical stress, not elsewhere classified"


# Transformation Tasks

### 1. Create a summary of how many times each patient received a specific vaccine (e.g., Seasonal Flu Vaccine, COVID-19 Vaccine).

In [0]:


patient_vaccine_summary = new_immunization_df.join(new_patients_df,new_immunization_df.vaccined_patient_id == new_patients_df.patient_id,"right").groupBy('vaccined_patient_id','first_name','description').count().orderBy(col("count").desc())

patient_vaccine_summary.withColumnRenamed('count','TotalNumberOfVaccine').limit(20).display()

vaccined_patient_id,first_name,description,TotalNumberOfVaccine
48fab85f-3db1-0093-9b08-4bc1480f98e4,Suzann,Seasonal Flu Vaccine,13
0d3ce9d4-dbbe-a8b8-5c8e-b85fd04ba2fa,Alessandra,Seasonal Flu Vaccine,13
728d585b-ceee-a8ba-543d-088f0aa67d77,Kristal,Seasonal Flu Vaccine,13
a57c3081-e4ee-e79d-f271-a088f490fefd,Su,Seasonal Flu Vaccine,13
77774c87-e69f-8e56-8e8c-3c59f1825c7c,Lilliam,Seasonal Flu Vaccine,13
5a9a010e-6e8a-b4b1-083c-37353ecd7325,Rosalva,Seasonal Flu Vaccine,12
8fcd3c7d-c0d9-b530-bba8-e8894f647d02,Chelsea,Seasonal Flu Vaccine,12
86672e83-f379-53d4-3eb6-62d45f9fa3ec,Silva,Seasonal Flu Vaccine,12
4aaacfe8-ced5-6a81-f02f-1cd576e13ba7,Jeffie,Seasonal Flu Vaccine,12
01c61f37-3878-f146-1eae-e0b159592672,Barbara,Seasonal Flu Vaccine,12


### 2. Flag Patients with Chronic Conditions

In [0]:
# Note the ChroniCc State is measured according to the stop date of the condition.
chronic_condition = new_conditions_df.withColumn("condition_state", 
                                                                when(new_conditions_df.stop_date.isNull(),lit("Chronic"))
                                                                .otherwise(lit("Non Chronic")))

chronic_condition.limit(10).display()

start_date,stop_date,patient_id,encounter_id,code,description,condition_state
1953-06-03,null,531997f3-3373-058b-6315-bab2b9f05502,b67eb71d-01d4-e225-7203-8d226d9079de,473.8,Other chronic sinusitis,Chronic
1952-03-12,null,12556183-9867-11ec-e7d6-4cc149e28bf6,56c1ae8b-c774-0259-9291-782e502027e4,473.9,Unspecified sinusitis (chronic),Chronic
1957-10-09,null,82e05bd5-78f8-2fe4-caba-33c4afdcc9ef,5f37acfc-84c0-7bac-284f-2b8fd9ef61ac,V60.9,Unspecified housing or economic circumstance,Chronic
1954-02-13,1956-02-25,63dc3aeb-c77c-441a-ffbd-a4d6985fbd77,4c822b22-96ee-f968-e29a-0227af97c084,V62.89,"Other psychological or physical stress, not elsewhere classified",Non Chronic
1951-09-13,null,d70308e9-5bce-1154-cefe-27f61aede16d,9967cce2-0a85-9dcd-92f1-52aa0fee7d7a,541,"Appendicitis, unqualified",Chronic
1959-09-29,null,59ce02b3-e5d8-da4c-2f70-b7018a853333,a268dba0-dd76-3c9f-8afc-b6861e9e7c37,473.8,Other chronic sinusitis,Chronic
1958-10-01,1959-04-30,5799e37d-3ae4-bd7b-61e2-8f4d6fb90956,f541a806-dd26-a3cd-478d-d57274d7a996,V22.2,"Pregnant state, incidental",Non Chronic
1959-12-22,null,9b0886ff-99fd-c2db-7d0c-7c25f8ee8636,b2690c76-b9e7-e737-57fe-33106b48ffb0,542,Other appendicitis,Chronic
1949-02-15,null,dc415ea3-e2d6-3808-ec78-79460d1ad67e,97b70d55-5289-67c8-73b8-f612963b7f75,V85.30,"Body Mass Index 30.0-30.9, adult",Chronic
1955-09-30,1956-05-04,5fbb2718-d088-9217-abe8-cefb053869ac,ea804eed-1031-20cd-9ea4-f86cdd2e9b53,V62.89,"Other psychological or physical stress, not elsewhere classified",Non Chronic


### Identify Patients with No Vaccinations

In [0]:
# Getting essential columns 
# new_patient = new_patients_df.select('patient_id','first_name','last_name').distinct()

# renaming columns for differentiation
new_immunization = new_immunization_df.select('vaccined_patient_id','description').distinct()

#Used Left_anti join for getting only the patient data with vaccination data
result_no_vacc_patient = new_patients_df.join(new_immunization,new_patients_df.patient_id == new_immunization.vaccined_patient_id,"left_anti")

result_no_vacc_patient.display()


patient_id,birthdate,deathdate,ssn,drivers,passport,prefix,first_name,last_name,suffix,maiden,marital,race,ethnicity,gender,birthplace,address,city,state,county,fips,zip,latitude,longitutde,healthcare_expenses,healthcare_coverage,income,mrn
9e8f1aa3-2e74-6a32-1f1b-21cc276be427,1919-02-07,1929-03-26,999-79-8799,NULL,NULL,NULL,Cleopatra,Abshire,NULL,NULL,n/a,white,hispanic,Female,Waltham Massachusetts US,467 Walker Way Unit 85,Boston,Massachusetts,Suffolk County,25025,2130,42.40268869877832,-71.08477134543801,23644.94,23900.71,113399,3971
5e742a3c-8c06-20c3-ede8-82d26d30d963,1926-04-02,1944-11-07,999-27-5800,S99910940,NULL,Ms.,Isidra,Mosciski,NULL,NULL,n/a,white,nonhispanic,Female,Somerville Massachusetts US,498 Schamberger Dam Apt 24,Raynham,Massachusetts,Bristol County,null,0,41.97276223246595,-71.03970841202607,314658.56,101971.68,71435,5054
94763640-ebae-65fd-6343-74b325d7c31a,1930-12-14,1937-08-20,999-91-9325,NULL,NULL,NULL,Shawn,Mueller,NULL,NULL,n/a,white,nonhispanic,Female,Huntington Massachusetts US,116 Kozey Esplanade,Southborough,Massachusetts,Worcester County,null,0,42.2655112888594,-71.5375049079105,14889.99,14208.94,193818,5243
cd1b7e2e-c688-1ec5-7441-864058b890c2,1913-09-28,1936-03-11,999-68-5072,S99999805,X10965565X,Ms.,Lottie,Heller,NULL,NULL,n/a,white,nonhispanic,Female,Maynard Massachusetts US,376 Considine Crossing Apt 62,Sandwich,Massachusetts,Barnstable County,25001,2563,41.75924071110232,-70.49618203610127,61407.72,0.0,85541,6999
7b4fea77-9563-b0ee-f3a9-6295fb8ced2b,1915-12-26,1941-05-21,999-63-9419,S99923915,X52212810X,Ms.,Blake,Anderson,NULL,NULL,n/a,white,nonhispanic,Female,Cambridge Massachusetts US,402 Senger Gate,New Bedford,Massachusetts,Bristol County,25005,2744,41.56773304550542,-70.92682120089707,5139.74,498968.17,1521,7518
60daced8-5fb0-9868-fab1-4a9fff305faf,1913-10-12,1943-09-07,999-29-2312,S99979878,X89135274X,Mrs.,Jolie,Volkman,NULL,Mertz,Married,white,nonhispanic,Female,Bernardston Massachusetts US,553 Koelpin Harbor Suite 89,Andover,Massachusetts,Essex County,25009,1810,42.68572249945525,-71.20073885340706,602784.53,148059.03,120427,7769
36eb46fa-7a40-298e-5723-3e92cef08635,1916-11-02,1930-06-10,999-59-2036,NULL,NULL,NULL,Bernita,Turner,NULL,NULL,n/a,asian,nonhispanic,Female,Brockton Massachusetts US,768 Hilpert Mall Unit 85,Weymouth,Massachusetts,Norfolk County,25021,2188,42.21913682202356,-70.92392987803692,17061.63,0.0,109372,9471
7c84dbb6-7b97-9424-f914-7de88381e9fe,1927-05-19,1931-08-13,999-23-3407,NULL,NULL,NULL,Janina,Medhurst,NULL,NULL,n/a,black,nonhispanic,Female,Norton Massachusetts US,679 Volkman Rest Suite 9,Plymouth,Massachusetts,Plymouth County,25023,2360,41.86081643341036,-70.67542736185756,7745.85,0.0,153618,10038
